In [NumPy Fundamentals](numpy_fundamentals.ipynb), you learned to reason about
array shapes, broadcasting, and axes, and in the pandas chapters before it you
learned to build, transform, and summarize labeled tables. In practice you
rarely use one library alone. A working analysis reads and cleans a labeled
table with pandas, hands a well-defined block of numbers to NumPy when an array
operation expresses the calculation better, and then returns the result to
pandas so it can be labeled, checked, and reported.

This chapter walks that path one stage at a time. Each stage gets its own small,
self-contained example, so you can see what each library is responsible for and
what has to be true at the two moments where data crosses between them.

By the end of this chapter you should be able to:

- Explain why an already-vectorized pandas operation is *already* running NumPy code.
- Audit and clean a raw column so that the numbers you hand to NumPy mean what you think they mean.
- Convert selected columns with `.to_numpy()` while keeping track of which record each row is.
- Compute with element-wise arithmetic, `np.where()`, `np.select()`, broadcasting, `axis` aggregations, and `@`.
- Return results to pandas as labeled Series and DataFrames, and explain why an array assigns by position while a Series assigns by label.
- Validate a result against a pandas reference, checking labels and missing values and not only shape.
- Measure the complete workflow with `%timeit`, including the conversion cost, and decide whether converting was worth it.

Complete the [practice activity](#practice-activity-from-messy-file-to-labeled-report)
after the worked stages.

## Set Up Your Chapter Files {#set-up-your-chapter-files}

Download the [Pandas and NumPy Workflow practice kit](downloads/pandas-numpy-workflow-practice.zip).
Extract `stat303-pandas-numpy-workflow` inside the `stat303-setup` project from the
setup chapters and select that project's verified Python environment.

```text
stat303-setup/
├── .venv/
└── stat303-pandas-numpy-workflow/
    ├── workflow_examples.ipynb
    ├── activity07.ipynb
    ├── README.md
    └── data/
        └── store_transactions.csv
```

Run `workflow_examples.ipynb` for the lesson and complete `activity07.ipynb` for
your own activity report. Use `stat303-pandas-numpy-workflow` as the notebook
working folder. The worked stages use `store_transactions.csv`; the activity
notebook writes and builds its own inputs in its first cell.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

print('numpy :', np.__version__)
print('pandas:', pd.__version__)

numpy : 2.5.3
pandas: 3.0.5


**Environment check:** This chapter uses NumPy and pandas together. If an import
fails, check the selected notebook kernel first. If a package is missing, activate
your project environment and run this command in its terminal:

```bash
python -m pip install numpy pandas
```

## One Stack, Two Layers

With both libraries available in your environment, start with how they sit
relative to each other, because that relationship explains every recommendation
in this chapter. pandas and NumPy are not two competing choices for the same job.
They are two layers of one stack: NumPy holds the numbers, and pandas adds
everything a labeled table needs on top of them.

```text
                       your analysis
                             │
  ┌──────────────────────────▼──────────────────────────┐
  │  pandas                                             │
  │  labels, one dtype per column, missing values,      │
  │  dates, text, reading and writing files             │
  │  → load, clean, and report                          │
  └──────────────────────────┬──────────────────────────┘
                             │  each numeric column is one of these
  ┌──────────────────────────▼──────────────────────────┐
  │  NumPy                                              │
  │  one contiguous block of same-typed numbers,        │
  │  addressed by position                              │
  │  → arithmetic, broadcasting, axes, matrix math      │
  └─────────────────────────────────────────────────────┘
```

That picture is a literal description of how the objects are built. A pandas
`Series` is a NumPy array plus an index. A `DataFrame` is a collection of
columns, and each column with a single numeric dtype is stored as one contiguous
NumPy array. So when you write `df['units'] * df['unit_price']`, pandas is not
looping over rows in Python. It hands the two underlying arrays to NumPy's
compiled multiplication routine and wraps the resulting array back up with an
index.

Because the two libraries are layers rather than competitors, "pandas or NumPy?"
is not really a question about speed. It is a question about **what your data
looks like at this moment in the workflow**. Data arriving from a file is labeled
and mixed, which is pandas-shaped. Data in the middle of a calculation is a block
of same-typed numbers, which is NumPy-shaped. Results going to a reader need
names back on them, which is pandas-shaped again.

In [2]:
scores = pd.Series([88, 92, 79, 95, 84],
                   index=['Ana', 'Ben', 'Cleo', 'Dan', 'Eve'],
                   name='quiz1')

print(scores)
print()
print('Series type :', type(scores))
print('Array type  :', type(scores.to_numpy()))
print('Array dtype :', scores.to_numpy().dtype)
print('Array values:', scores.to_numpy())
print('Same memory :', np.shares_memory(scores, scores.to_numpy()))

Ana     88
Ben     92
Cleo    79
Dan     95
Eve     84
Name: quiz1, dtype: int64

Series type : <class 'pandas.Series'>
Array type  : <class 'numpy.ndarray'>
Array dtype : int64
Array values: [88 92 79 95 84]
Same memory : True


`scores.to_numpy()` hands back the numbers that were backing the Series — and
nothing else. The names `Ana`, `Ben`, `Cleo` are gone.

`np.shares_memory()`, from
[Views and Copies](numpy_fundamentals.ipynb#views-and-copies) in the NumPy
chapter, reports `True` here: nothing was copied, and the array you were handed
is the very block of memory the Series was already using. The two layers are not
two copies of your data. They are one block of numbers, with or without the
labels.

What a conversion costs you, then, is not the numbers. It is the labels — and
that is the cost the rest of this chapter is careful about paying.

## Vectorized pandas Is Already NumPy

Seeing the two layers share one block of memory settles a question students
usually ask at this point: if NumPy is the fast one, should every calculation be
converted? The measurement below says no, and it is worth taking before you learn
anything else about converting.

When a pandas expression works on whole columns at once, the fast path is already
in use. `df['units'] * df['unit_price']`, `df['units'] >= 20`, `df['units'].sum()`,
`.mean()`, `.max()` — each of these hands a complete column to compiled NumPy code
and gets an array back. There is no Python loop inside them left to remove.

The 24 records in the transactions file are far too few to time, so build a larger
table with the same structure. `%timeit` runs a line many times and reports an
average, which is far more trustworthy than a single run; outside Jupyter, use
`time.perf_counter()` before and after.

In [24]:
rng = np.random.default_rng(303)
n = 100_000

big = pd.DataFrame({
    'store': rng.choice(['North', 'South', 'West'], size=n),
    'product': rng.choice(['Notebook', 'Pen', 'Folder', 'Marker'], size=n),
    'units': rng.integers(1, 80, size=n),
    'unit_price': rng.choice([5.0, 2.0, 3.0, 4.0], size=n),
})
print(big.shape)
big.head()

(100000, 4)


,store,product,units,unit_price
0,South,Marker,46,5.0
1,North,Notebook,66,4.0
2,West,Pen,60,2.0
3,South,Pen,53,4.0
4,South,Marker,1,3.0


In [25]:
print('pandas Series arithmetic:')
%timeit big['units'] * big['unit_price']

print('the same thing, converted to arrays first:')
%timeit big['units'].to_numpy() * big['unit_price'].to_numpy()

pandas Series arithmetic:
58.4 μs ± 773 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
the same thing, converted to arrays first:
51.1 μs ± 698 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


Both timings land in the same neighborhood. Converting by hand shaved off a
little of pandas's per-operation bookkeeping — checking dtypes, aligning indexes,
wrapping the result back up in a Series — but the arithmetic itself was the same
compiled NumPy loop either way. The calculation did not get faster; it only lost
its labels.

Keep that as your default position for the rest of the chapter: **if your pandas
code is already vectorized, converting it to NumPy is not what makes it fast.**

## When Converting Is Worth It

So why convert at all? Three situations make it worth doing, and ordinary column
arithmetic is not one of them.

**The calculation is natural on a rectangle of numbers.** Collapsing an axis,
scaling every row by its own total, and multiplying a table of quantities by a
vector of prices are each one short expression on an array and each awkward on a
record-per-row table. The reason to convert here is clarity rather than speed:
the axis and the shape are visible in the code instead of hidden inside a method
name.

**Your own code has broken the vectorization.** A `.apply()` that calls a Python
function once per record puts the row loop back into Python, and that loop, not
the arithmetic, is where the time goes. Rewriting the rule with `np.where()` or
`np.select()` is the single change in this chapter that is worth hundreds of
times its own length.

**The same numbers are used over and over.** pandas's per-operation overhead is
small, but a loop that runs the same calculation for many parameter values pays
it on every pass. Convert once, outside the loop, and every pass after that is
plain array arithmetic.

Converting is not free. You give up the index, automatic label alignment, and
missing-value-aware aggregation. Read the middle column as "where this task
belongs," not "which library is better."

| Situation | Prefer | Why |
|---|---|---|
| Reading files, parsing dates, cleaning text, repairing dtypes | **pandas** | None of these operations exist for a homogeneous numeric block |
| Element-wise math or comparisons on numeric columns | **either; pandas is fine** | pandas hands the work to NumPy already, so converting changes little |
| A `.apply()` running a Python function per row | **rewrite as vectorized** | Removes the per-row Python loop; by far the largest win available |
| Broadcasting, matrix products, reshaping, sorting along an axis | **NumPy** | Positional, homogeneous work — both faster and clearer as arrays |
| The same computation repeated hundreds of times | **NumPy arrays** | pandas's small per-operation overhead adds up once multiplied by hundreds |
| Data with more than two dimensions | **NumPy** | A DataFrame is two-dimensional by construction |
| Data contain missing values | **pandas, or be careful** | `Series.mean()` skips `NaN`; `array.mean()` does not. Use `np.nanmean()` and friends, or settle on complete cases first |
| Rows were filtered or sorted after you pulled the array out | **re-pull, or assign back as a labeled Series** | Array assignment is positional, so a stale array attaches to the wrong records with no error |
| Output that a human will read | **pandas** | Labels, alignment, rounding, and a readable display |

A useful default: **stay in pandas unless the data in front of you is genuinely a
rectangle of same-typed numbers, in which case reach for NumPy and enjoy it.**

Several rows there are warnings rather than recommendations — missing values,
stale arrays — and each one gets a demonstration later, in the stage where it
does its damage. Converting safely is a procedure, and the rest of the chapter
walks through it.

## The Workflow in Seven Stages

Those three situations say when to cross between the libraries. The stages
below say how to cross without losing anything on the way. One analysis runs
through all of them, from the file on disk to a checked, labeled report.

1. **Read** the file with pandas.
2. **Inspect and clean** types, missing values, identifiers, and units.
3. **Convert** the numeric core you actually need, keeping the labels you will want back.
4. **Compute** with the NumPy operation that fits the calculation.
5. **Return** the result to pandas as labeled Series or DataFrames.
6. **Validate** against a reference you trust, including labels and missing values.
7. **Measure** the complete workflow and decide whether the conversion paid for itself.

Conversion is a decision inside this workflow, not a required step in every
analysis. Stages 1, 2, 6, and 7 live in pandas and stage 4 is NumPy's; notice the
shape of that list, because it is the stack diagram turned into a sequence —
**pandas at both ends, NumPy in the middle.**

Stages 3 and 5 are the conversion boundaries, and they are the only two moments
in the list where meaning can quietly leak away: you hand numbers to NumPy at
stage 3 and hand results back to pandas at stage 5. Everything on either side of
them is ordinary pandas or ordinary NumPy. Each boundary asks you to decide
something no library can decide for you:

| Stage | Boundary | What you must decide |
|---|---|---|
| 3 | pandas → NumPy | Which columns, in which row and column order, with what done about missing values |
| 5 | NumPy → pandas | Which index and column labels the result belongs to |

The rest of this chapter takes one stage at a time.

### Stage 1: Read with pandas

pandas reads files. NumPy can read numbers from a text file, but it has no good
answer for a table that mixes store names, dates, and dollar amounts — which is
what real files look like. Start in pandas, always.

The practice kit ships this table as `data/store_transactions.csv`. It records one
order line per row: a store, a product, a date, a channel, a quantity, and a unit
price. Read it the way you would read any file of your own.

In [3]:
sales = pd.read_csv(Path('data') / 'store_transactions.csv')

print('shape:', sales.shape)
sales

shape: (24, 7)


,order_id,store,product,order_date,channel,units,unit_price
0,1001,North,Notebook,2024-03-01,online,12,$5.00
1,1002,North,Pen,2024-03-01,store,60,$2.00
2,1003,North,Folder,2024-03-02,online,24,$3.00
3,1004,North,Marker,2024-03-02,store,18,$4.00
4,1005,South,Notebook,2024-03-03,online,10,$5.00
5,1006,South,Pen,2024-03-03,store,55,$2.00
6,1007,South,Folder,2024-03-04,online,20,$3.00
7,1008,South,Marker,2024-03-04,store,NaN,$4.00
8,1009,West,Notebook,2024-03-05,online,14,$5.00
9,1010,West,Pen,2024-03-05,store,unknown,$2.00


### Stage 2: Inspect and Clean

Before you trust a single number, look at what pandas actually handed you. A CSV
file carries no type information, so every column's dtype is a guess that
`read_csv` made from the text it saw. Check that guess and repair it here, while
you still have pandas's string and datetime tools — NumPy has nothing to offer a
column full of dollar signs. This is the stage students skip and then spend an
afternoon debugging.

In [4]:
print(sales.dtypes)

order_id      int64
store           str
product         str
order_date      str
channel         str
units           str
unit_price      str
dtype: object


Read the dtypes before you read the numbers. `order_id` arrived as an integer,
but `order_date`, `units`, and `unit_price` all arrived as text — reported here as
`str`, and as `object` by older pandas versions. Three columns that look like
numbers or dates are still strings.

The reasons are worth naming, because each one recurs constantly:

- `unit_price` carries a `$`, and a dollar sign is not part of a number.
- `units` contains the word `unknown` in one record. **One unusable cell in a
  million is enough to demote the entire column to text.**
- `order_date` is a perfectly reasonable date format that pandas will not parse
  unless you ask it to.

Convert this table to an array now and you get an array of strings. Some
arithmetic on it would raise an error, which is the good case. Some would
succeed and hand you nonsense: multiplying a text column by `2` repeats the
characters, so `'12'` becomes `'1212'` instead of `24`.

So repairing these dtypes is not cosmetic tidying. It is what makes the fast path
available at all.

In [5]:
clean = sales.copy()

# Strip currency formatting, then convert. errors='coerce' turns anything
# unreadable into NaN instead of raising.
price_text = (clean['unit_price']
              .str.replace('$', '', regex=False)
              .str.replace(',', '', regex=False))

clean['unit_price'] = pd.to_numeric(price_text, errors='coerce')
clean['units'] = pd.to_numeric(clean['units'], errors='coerce')
clean['order_date'] = pd.to_datetime(clean['order_date'])

print(clean.dtypes)

order_id               int64
store                    str
product                  str
order_date    datetime64[us]
channel                  str
units                float64
unit_price           float64
dtype: object


`errors='coerce'` is convenient and dangerous in the same breath: it converts
what it can and silently replaces the rest with `NaN`. Convenient, because one bad
cell does not stop the analysis. Dangerous, because a column that is 40% unreadable
looks exactly like a column that is 40% genuinely unrecorded. So audit the
coercion every time: count the missing values, and look at the original text
behind them.

One detail about what pandas does on its own: `read_csv` already recognizes some
spellings of "missing" — a blank field, and also `NA`, `N/A`, `null`, `NaN`, and
`n/a` — and turns them into `NaN` without being asked. What it cannot guess is an
arbitrary word like `unknown`, which is precisely why `errors='coerce'` exists.

In [6]:
print('missing values after cleaning:')
print(clean.isna().sum())

print('\nrecords whose units could not be read as a number:')
sales.loc[clean['units'].isna(), ['order_id', 'store', 'product', 'units']]

missing values after cleaning:
order_id      0
store         0
product       0
order_date    0
channel       0
units         2
unit_price    0
dtype: int64

records whose units could not be read as a number:


,order_id,store,product,units
7,1008,South,Marker,NaN
9,1010,West,Pen,unknown


Two records lost their `units`, and the original text tells you they are not the
same kind of problem. Order 1008 was left blank — the quantity was never recorded.
Order 1010 says `unknown` — someone recorded that they did not know. Both arrive as
`NaN`, and pandas cannot tell them apart for you. Deciding what each one means is
your job, not the library's, and it belongs in writing next to the code.

#### Missing Values Change Answers at the Boundary

pandas and NumPy disagree, by design, about what an aggregation should do when a
value is missing. pandas skips missing values; a plain NumPy array does not. This
is the single most common way a conversion silently changes a result.

In [7]:
units_col = clean['units']

print('pandas Series .mean()  :', units_col.mean())
print('numpy array .mean()    :', units_col.to_numpy().mean())
print('numpy np.nanmean()     :', np.nanmean(units_col.to_numpy()))
print()
print('records in column      :', len(units_col))
print('observed (non-missing) :', units_col.notna().sum())

pandas Series .mean()  : 26.727272727272727
numpy array .mean()    : nan
numpy np.nanmean()     : 26.727272727272727

records in column      : 24
observed (non-missing) : 22


`units_col.mean()` averaged the 22 observed quantities. `units_col.to_numpy().mean()`
returned `nan`, because in NumPy any arithmetic touching `nan` produces `nan`.
`np.nanmean()` skips the missing values and matches pandas again.

The `nan` at least announces itself. The version that should worry you is a sum or
a count where the missing values quietly become zeros, or a denominator that
includes records you never observed. Before converting, decide explicitly what
happens to incomplete records and say so.

Here the decision is to analyze complete cases only, and to report how many
records that excludes. With the missing values gone, `units` can go back to an
integer dtype — which it could not do a moment ago, because a NumPy integer array
has no value to represent "missing."

In [8]:
# One mask marking every record that dropna is about to remove, so the report
# below and the filter below it can never disagree.
incomplete = clean['units'].isna() | clean['unit_price'].isna()

analysis = clean.dropna(subset=['units', 'unit_price']).copy()
analysis['units'] = analysis['units'].astype('int64')   # safe now: no NaN left

print('kept', len(analysis), 'of', len(clean), 'records')
print('excluded order_ids:', sales.loc[incomplete, 'order_id'].tolist())
print()
print(analysis.dtypes)

kept 22 of 24 records
excluded order_ids: [1008, 1010]

order_id               int64
store                    str
product                  str
order_date    datetime64[us]
channel                  str
units                  int64
unit_price           float64
dtype: object


### Stage 3: Convert the Numeric Core

Convert the columns you are about to compute with — not the whole table. A
DataFrame that mixes store names with quantities has no single dtype that fits
everything, so pandas falls back to a general-purpose `object` array. That array
is a NumPy array in name only: it holds pointers to Python objects, gives up the
compiled fast paths, and is not what you want to do math on.

In [9]:
numeric_block = analysis[['units', 'unit_price']].to_numpy()
print('selected numeric columns ->', numeric_block.dtype, numeric_block.shape)
print(numeric_block[:4])

whole_table = analysis.to_numpy()
print('\nwhole DataFrame          ->', whole_table.dtype, whole_table.shape)
print(whole_table[0])

selected numeric columns -> float64 (22, 2)
[[12.  5.]
 [60.  2.]
 [24.  3.]
 [18.  4.]]

whole DataFrame          -> object (22, 7)
[1001 'North' 'Notebook' Timestamp('2024-03-01 00:00:00') 'online' 12 5.0]


**Rule of thumb:** convert the specific numeric columns you plan to compute with,
and only convert the whole frame when every column already shares one numeric
dtype. Check `.dtype` immediately after converting — if it reads `object`, stop
and fix the selection.

You will also meet `.values` doing this job in older code. Prefer `.to_numpy()` in
anything you write: `.values` behaves inconsistently for specialized column types
such as categorical or timezone-aware datetime columns, and `.to_numpy()` is the
documented replacement.

The other half of this stage is bookkeeping. The array has no index, so before you
leave pandas, hold on to whatever you will need to put the answer back — the index
itself, and a genuine record identifier if you have one. Keeping the identifier is
what lets you *prove* later that row 7 of your result is still order 1009, rather
than assuming it.

In [10]:
row_index = analysis.index                    # index labels of the kept records
order_ids = analysis['order_id'].to_numpy()   # the real identifier

print('index    :', row_index[:6].tolist())
print('order_ids:', order_ids[:6])

index    : [0, 1, 2, 3, 4, 5]
order_ids: [1001 1002 1003 1004 1005 1006]


### Stage 4: Compute with NumPy

Now the arrays are clean and you know what each row is. This is where NumPy earns
its place.

#### Element-wise Arithmetic and Conditional Values

A tiered volume discount is exactly the kind of rule people write as a row
function and a chain of `if`/`elif`. `np.select()` says the same thing as one
vectorized expression: a list of conditions, a matching list of values, and a
default for everything else. `np.where()` is its two-outcome special case.

In [11]:
units = analysis['units'].to_numpy()
price = analysis['unit_price'].to_numpy()

gross = units * price

# 50+ units -> 10% off; 20-49 units -> 5% off; fewer -> no discount.
discount_rate = np.select([units >= 50, units >= 20],
                          [0.10, 0.05],
                          default=0.0)
revenue = gross * (1 - discount_rate)

# np.where: one condition, two outcomes. Small orders pay a handling fee.
handling = np.where(revenue < 75, 4.95, 0.0)
billed = revenue + handling

print('units    :', units[:6])
print('gross    :', gross[:6])
print('rate     :', discount_rate[:6])
print('revenue  :', revenue[:6])
print('handling :', handling[:6])
print('billed   :', billed[:6])

units    : [12 60 24 18 10 55]
gross    : [ 60. 120.  72.  72.  50. 110.]
rate     : [0.   0.1  0.05 0.   0.   0.1 ]
revenue  : [ 60.  108.   68.4  72.   50.   99. ]
handling : [4.95 0.   4.95 4.95 4.95 0.  ]
billed   : [ 64.95 108.    73.35  76.95  54.95  99.  ]


`np.select()` checks its conditions in order and takes the first match, which is
why `units >= 50` must come before `units >= 20`. Reverse the two and every large
order would be handed the 5% tier — with no error to tell you.

#### A Rectangle of Numbers: Axes and Broadcasting

The other half of Stage 4 is the work that is natural on a *rectangle* of numbers
and awkward on a record-per-row table. Here is a small weekly summary: one row per
day, one column per product, values are units sold. Every column is the same
dtype, so it forms a clean numeric block.

In [12]:
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']
products = ['Notebook', 'Pen', 'Folder', 'Marker']

weekly = pd.DataFrame(
    [[120,  37,  36,  30],
     [ 59, 160, 150,  44],
     [ 83,  60, 100, 168],
     [136, 186,  41, 198],
     [117,  86, 148,  20]],
    index=days,
    columns=products,
)
weekly

,Notebook,Pen,Folder,Marker
Mon,120,37,36,30
Tue,59,160,150,44
Wed,83,60,100,168
Thu,136,186,41,198
Fri,117,86,148,20


In [13]:
mat = weekly.to_numpy()
print('dtype:', mat.dtype, '| shape:', mat.shape)
mat

dtype: int64 | shape: (5, 4)


array([[120,  37,  36,  30],
       [ 59, 160, 150,  44],
       [ 83,  60, 100, 168],
       [136, 186,  41, 198],
       [117,  86, 148,  20]])

**Totals along each axis.** The axis you name is the one that *disappears*.
`axis=1` collapses the columns and leaves one total per day; `axis=0` collapses the
rows and leaves one total per product.

In [14]:
per_day = mat.sum(axis=1)       # (5,) -- one number per row
per_product = mat.sum(axis=0)   # (4,) -- one number per column

print('units per day    :', per_day, per_day.shape)
print('units per product:', per_product, per_product.shape)

units per day    : [223 413 411 561 371] (5,)
units per product: [515 529 475 460] (4,)


**Each product's share of its own day's total.** This is the "one factor per row"
broadcasting pattern. `per_day` has shape `(5,)`, which will not line up against a
`(5, 4)` matrix. Giving it shape `(5, 1)` makes it a column vector, and NumPy then
reuses each day's total across that day's four products without copying anything.

In [15]:
row_totals = mat.sum(axis=1, keepdims=True)   # (5, 1); same as per_day.reshape(-1, 1)
share = mat / row_totals

print('shapes:', mat.shape, '/', row_totals.shape, '->', share.shape)
print('every row sums to 1?', np.allclose(share.sum(axis=1), 1.0))
share.round(3)

shapes: (5, 4) / (5, 1) -> (5, 4)
every row sums to 1? True


array([[0.538, 0.166, 0.161, 0.135],
       [0.143, 0.387, 0.363, 0.107],
       [0.202, 0.146, 0.243, 0.409],
       [0.242, 0.332, 0.073, 0.353],
       [0.315, 0.232, 0.399, 0.054]])

**Revenue per day with a matrix product.** With one price per product, revenue per
day is "multiply each day's units by the price vector and add them up" — which is
exactly what `@` does. One symbol replaces a nested loop.

`@` pairs the array's columns with the vector's entries **by position, not by
name**, so the price order has to match the column order. Compare the two
deliberately before converting.

In [16]:
print('column order:', weekly.columns.tolist())
prices = np.array([5.0, 2.0, 3.0, 4.0])      # Notebook, Pen, Folder, Marker

revenue_per_day = mat @ prices               # (5, 4) @ (4,) -> (5,)

print('shapes     :', mat.shape, '@', prices.shape, '->', revenue_per_day.shape)
print('revenue    :', revenue_per_day)
print('matches an explicit broadcast-and-sum?',
      np.allclose(revenue_per_day, (mat * prices).sum(axis=1)))

column order: ['Notebook', 'Pen', 'Folder', 'Marker']
shapes     : (5, 4) @ (4,) -> (5,)
revenue    : [ 902. 1241. 1507. 1967. 1281.]
matches an explicit broadcast-and-sum? True


Shuffle the price vector and this still returns five perfectly plausible dollar
amounts, with every notebook priced as a pen. Nothing raises an error. A shape
check would pass. The only protection is having compared the two orders yourself.

### Stage 5: Return Labeled Results

A NumPy array is an answer without a subject. Put the labels back immediately,
while you still remember what the axes meant.

Start with the record-level results. `revenue` and `billed` hold one value per kept
transaction, in the row order of `analysis`, so wrapping them in a `pd.Series` with
the index you saved in Stage 3 attaches every number to the record it came from.

In [17]:
revenue_series = pd.Series(revenue, index=row_index, name='revenue')
billed_series = pd.Series(billed, index=row_index, name='billed')

analysis['revenue'] = revenue_series
analysis['billed'] = billed_series

analysis[['order_id', 'store', 'product', 'units', 'unit_price',
          'revenue', 'billed']].head()

,order_id,store,product,units,unit_price,revenue,billed
0,1001,North,Notebook,12,5.0,60.0,64.95
1,1002,North,Pen,60,2.0,108.0,108.00
2,1003,North,Folder,24,3.0,68.4,73.35
3,1004,North,Marker,18,4.0,72.0,76.95
4,1005,South,Notebook,10,5.0,50.0,54.95


The rectangle results from the weekly table need labels on **both** axes. When you
rebuild a DataFrame from a 2D array, pass the original `index` and `columns` so
each number lands under the right name; when the array holds one value per row, it
becomes a new column on the table it came from.

In [18]:
share_report = pd.DataFrame(share, index=weekly.index, columns=weekly.columns)
share_report.round(3)

,Notebook,Pen,Folder,Marker
Mon,0.538,0.166,0.161,0.135
Tue,0.143,0.387,0.363,0.107
Wed,0.202,0.146,0.243,0.409
Thu,0.242,0.332,0.073,0.353
Fri,0.315,0.232,0.399,0.054


In [19]:
# One value per row -> a new column on the existing table.
daily_report = weekly.copy()
daily_report['units_total'] = per_day
daily_report['revenue'] = revenue_per_day

daily_report

,Notebook,Pen,Folder,Marker,units_total,revenue
Mon,120,37,36,30,223,902.0
Tue,59,160,150,44,413,1241.0
Wed,83,60,100,168,411,1507.0
Thu,136,186,41,198,561,1967.0
Fri,117,86,148,20,371,1281.0


#### An Array Assigns by Position; a Series Assigns by Label

This is the most expensive difference in the chapter. When you assign a plain
NumPy array into a DataFrame, pandas lines it up by **row position**. When you
assign a Series, pandas lines it up by **index label**. As long as nothing has
been sorted or filtered, the two agree. Once the row order changes, they do not —
and the positional version fails silently.

In [20]:
by_units = analysis.sort_values('units')    # row order now differs from `analysis`

by_units['revenue_positional'] = revenue          # array -> matched by POSITION
by_units['revenue_labelled'] = revenue_series     # Series -> matched by LABEL

by_units[['order_id', 'units', 'unit_price',
          'revenue_positional', 'revenue_labelled']].head()

,order_id,units,unit_price,revenue_positional,revenue_labelled
4,1005,10,5.0,60.0,50.0
22,1023,11,3.0,108.0,33.0
0,1001,12,5.0,68.4,60.0
14,1015,12,3.0,72.0,36.0
8,1009,14,5.0,50.0,70.0


In [21]:
wrong = (by_units['revenue_positional'] != by_units['revenue_labelled']).sum()
print(wrong, 'of', len(by_units), 'records received the wrong revenue')
print('no error was raised, and both columns have the right length and dtype')

# The identifier saved in Stage 3 is what makes the misalignment visible.
print()
print('row order still matches the array?',
      np.array_equal(order_ids, by_units['order_id'].to_numpy()))

21 of 22 records received the wrong revenue
no error was raised, and both columns have the right length and dtype

row order still matches the array? False


Both assignments produced a full column of believable dollar amounts, and only
one of them is right. The positional version handed order 1005 the `60.0` that
belongs to order 1001, and shifted almost every other record the same way.

That last check is what Stage 3's bookkeeping was for. Comparing the saved
`order_ids` against the identifiers in the frame you are about to assign into
answers one question — "is this still the row order my array came from?" — and
here it answers no. Two habits keep you out of the situation to begin with:

- Do the convert → compute → assign-back sequence without reordering or filtering the DataFrame in between. That is why the `daily_report` assignment above was safe: nothing had touched `weekly`'s row order since `mat` was extracted from it.
- When you cannot promise that, wrap the result in a `pd.Series(values, index=...)` with the index you took the values from, and let pandas align by label.

One cheap check before any positional assignment is `len(arr) == len(df)`. It
catches a length mismatch but not a reordering — only the habits above, and the
identifier comparison, catch that.

### Stage 6: Validate Against a Reference

Write the obvious, slow, clearly correct version first, then check the fast one
against it. For a workflow that crossed the pandas/NumPy boundary twice, a
worthwhile check covers three things: the values, the labels, and the missing
values. The reference has to reproduce the *whole* rule — discount tiers and
handling fee together — or it only validates the part you already trusted.

In [22]:
def billed_row(row):
    """The obvious row-at-a-time version, used only as a reference."""
    if row['units'] >= 50:
        rate = 0.10
    elif row['units'] >= 20:
        rate = 0.05
    else:
        rate = 0.0
    row_revenue = row['units'] * row['unit_price'] * (1 - rate)
    fee = 4.95 if row_revenue < 75 else 0.0
    return row_revenue + fee


reference = analysis.apply(billed_row, axis=1)

print('values agree  :', np.allclose(reference.to_numpy(),
                                     analysis['billed'].to_numpy()))
print('labels agree  :', reference.index.equals(analysis['billed'].index))
print('missing counts:', reference.isna().sum(), analysis['billed'].isna().sum())

values agree  : True
labels agree  : True
missing counts: 0 0


Use `np.allclose()` for floating-point results and `(a == b).all()` or
`np.array_equal()` for values that should match exactly, such as integers,
booleans, or strings. Floating-point arithmetic can differ in the last bits
depending on the order of operations, so an exact `==` on floats fails for reasons
that have nothing to do with your logic.

#### A Shape Check Is Not Enough

"Same shape" is the weakest possible evidence. Here is the same array, shuffled:
identical length, identical dtype, completely wrong answer.

In [23]:
shuffled = analysis['revenue'].sample(frac=1, random_state=303).to_numpy()
truth = analysis['revenue'].to_numpy()

print('same shape :', shuffled.shape == truth.shape)
print('same dtype :', shuffled.dtype == truth.dtype)
print('same total :', np.isclose(shuffled.sum(), truth.sum()))
print('allclose   :', np.allclose(shuffled, truth))

same shape : True
same dtype : True
same total : True
allclose   : False


Even the total agrees, because a shuffle preserves the sum. Only the
element-by-element comparison catches it. Any summary statistic that is invariant
to row order — a sum, a mean, a count — cannot detect a misalignment, which is
exactly the failure mode that Stage 5 showed is easy to cause.

### Stage 7: Measure the Whole Workflow

The report is correct and its labels have been checked, so now it is fair to ask
whether the conversion paid for itself.
[Vectorized pandas Is Already NumPy](#vectorized-pandas-is-already-numpy) timed a
single expression and found no real difference. This stage times the rule this
chapter actually built — discount tiers and handling fee together — written once
as a row function passed to `.apply()` and once as a vectorized function, on the
same large `big` table.

The vectorized version makes its own `.to_numpy()` calls, so the measurement
covers the **whole workflow**, conversion included, rather than the arithmetic
alone.

In [26]:
def billed_vectorized(df):
    units = df['units'].to_numpy()
    price = df['unit_price'].to_numpy()
    rate = np.select([units >= 50, units >= 20], [0.10, 0.05], default=0.0)
    revenue = units * price * (1 - rate)
    return revenue + np.where(revenue < 75, 4.95, 0.0)


print('row-at-a-time with .apply(axis=1):')
%timeit big.apply(billed_row, axis=1)

print('vectorized, conversion included:')
%timeit billed_vectorized(big)

row-at-a-time with .apply(axis=1):
243 ms ± 3.03 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
vectorized, conversion included:
411 μs ± 9.45 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [27]:
# Correctness before speed: the measurement is meaningless if the answers differ.
print('results agree:', np.allclose(big.apply(billed_row, axis=1).to_numpy(),
                                    billed_vectorized(big)))

results agree: True


The gap is large — several hundredfold on this table — and it comes from one thing
only: where the row-by-row looping happens. `.apply(axis=1)` calls a Python
function once per record and packs each record into a small `Series` first, so
100,000 records cost 100,000 function calls and 400,000 label lookups. The
vectorized version loops once, inside compiled code, over the whole array.

Read this measurement together with the one in
[Vectorized pandas Is Already NumPy](#vectorized-pandas-is-already-numpy) before
you draw a conclusion from either. The win came from **removing the per-row Python
loop**, not from converting to arrays: a vectorized pandas expression ran at
essentially the same speed as the array version, so the conversion here is a
tune-up on top of the optimization rather than the optimization itself.

Two honest caveats about any number you get here:

- Times vary by machine, by dtype, and by table size. Measure your own workflow rather than trusting a remembered ratio.
- Measure the workflow you would actually ship. A conversion that saves 5 ms of arithmetic but adds 30 ms of cleaning and copying has made your analysis slower.

## Putting It Together

One function, all seven stages, with the checks written in rather than bolted on
afterwards. This is the shape worth copying into your own work: the conversion is
narrow and deliberate, the labels come straight back, and nothing is returned
until it has been compared against a reference.

In [28]:
def daily_revenue_report(raw_frame):
    """Stages 1-6 for one transactions table. Returns a labeled, validated report."""
    # Stage 2: clean, and be explicit about incomplete records.
    df = raw_frame.copy()
    price_text = (df['unit_price']
                  .str.replace('$', '', regex=False)
                  .str.replace(',', '', regex=False))
    df['unit_price'] = pd.to_numeric(price_text, errors='coerce')
    df['units'] = pd.to_numeric(df['units'], errors='coerce')
    df['order_date'] = pd.to_datetime(df['order_date'])

    incomplete = df['units'].isna() | df['unit_price'].isna()
    df = df[~incomplete].copy()
    df['units'] = df['units'].astype('int64')

    # Stage 3: convert only the numeric core, keeping the index.
    keep_index = df.index
    u = df['units'].to_numpy()
    p = df['unit_price'].to_numpy()

    # Stage 4: compute.
    rate = np.select([u >= 50, u >= 20], [0.10, 0.05], default=0.0)
    values = u * p * (1 - rate)
    with_fee = values + np.where(values < 75, 4.95, 0.0)

    # Stage 5: return labels.
    df['revenue'] = pd.Series(values, index=keep_index)
    df['billed'] = pd.Series(with_fee, index=keep_index)

    # Stage 6: validate against the slow-but-obvious version.
    ref = df.apply(billed_row, axis=1)
    assert np.allclose(ref.to_numpy(), df['billed'].to_numpy())
    assert ref.index.equals(df['billed'].index)

    print(f'{len(df)} records reported, {int(incomplete.sum())} excluded as incomplete')
    return df[['order_id', 'store', 'product', 'units', 'unit_price',
               'revenue', 'billed']]


report = daily_revenue_report(sales)
report.head()

22 records reported, 2 excluded as incomplete


,order_id,store,product,units,unit_price,revenue,billed
0,1001,North,Notebook,12,5.0,60.0,64.95
1,1002,North,Pen,60,2.0,108.0,108.00
2,1003,North,Folder,24,3.0,68.4,73.35
3,1004,North,Marker,18,4.0,72.0,76.95
4,1005,South,Notebook,10,5.0,50.0,54.95


Read the result as a description of these 22 records and this March window. It is
not an estimate of what any store sells in general, and the discount rule is an
invented teaching example.

## Practice Activity: From Messy File to Labeled Report {#practice-activity-from-messy-file-to-labeled-report}

**Goal:** Run one small dataset through all the stages of the workflow and defend each boundary crossing.

**File:** `activity07.ipynb`.

**Submit:** `activity07.html` through the final upload question in the Pandas and NumPy Workflow Canvas quiz.

**These are the complete Activity 7 instructions.** Use this section for the task
list; the notebook contains spaces to record your work.

Open `activity07.ipynb` from the folder prepared in
[Set Up Your Chapter Files](#set-up-your-chapter-files), using the same project
environment. Its first cell writes a messy CSV (`inventory_raw.csv`), builds a
large `orders` table for timing, and defines a small `warehouse` table with a
matching `unit_costs` vector. Run that cell before starting.

The worked stages above are preparation, not additional submission requirements.

### A. Load, Inspect, Clean {.unnumbered}

- Replace `Your Name` in the opening Raw cell's `author` field. Run the supplied setup cell; it must report `True` for the file check.
- Read `inventory_raw.csv` and display `.dtypes`. For each column that came in as text rather than as a number or a date, say in one sentence *why* it did.
- Repair all three problem columns so that `quantity` ends as an integer dtype, `cost` as a float dtype, and `restocked` as a datetime dtype. Strip the currency symbol before converting, and use `errors='coerce'`.
- Report how many missing values existed, display the **original** text behind each one, and say whether each reads as "never recorded" or "recorded as unknown."
- State what you chose to do about the incomplete records with a one-sentence justification, apply it, and explain why `quantity` could only return to an integer dtype after that step.

### B. Derive a Column Two Ways {.unnumbered}

- The rule: `value = quantity × cost`; then a bulk discount of 15% off when `quantity >= 40` and 8% off when `quantity >= 15`, and nothing below that; then a \$12.50 handling fee on any item whose discounted value is under \$200.
- Write it first as a row function and apply it to the large `orders` table with `.apply(axis=1)`. Time it with `%timeit`.
- Write it again vectorized, using `np.select()` for the tiers and `np.where()` for the handling fee. Time that too.
- Verify the two agree with `np.allclose`, then report the speedup as a ratio. Explain which change produced the speedup, and why the order of your `np.select` conditions matters.

### C. Hand Off to NumPy {.unnumbered}

- Convert the four numeric item columns of the `warehouse` table to a single 2D array. Display the `dtype` and confirm it is **not** `object`. Explain what you would have got by converting the whole table instead, and why that result would be useless for computation.
- Compute each warehouse's total (one number per row) and each item's total (one number per column). Name the axis you used in each case and explain why.
- Using broadcasting, compute each cell's share of its own **row** total. State the shapes involved, and verify every row of your result sums to 1.
- Using `@` and the supplied `unit_costs` vector, compute the total inventory value per warehouse. State the shapes of both inputs and of the result. Show the check you performed to confirm that `unit_costs` is in the same order as your array's columns, and explain what the result would have looked like if it had not been.

### D. Return to pandas {.unnumbered}

- Rebuild your share matrix as a DataFrame carrying the original row and column labels, rounded to three decimals.
- Add your per-warehouse value back onto the `warehouse` table as a new column, and explain in one or two sentences why the positional assignment is safe *in this specific case*.

### E. One Trap, Demonstrated {.unnumbered}

- Choose either the missing-value trap (`Series.mean()` versus `array.mean()`) or the positional-alignment trap (assigning an array back after sorting).
- Write a short cell that produces the wrong answer, then the corrected version, with the wrong and right values displayed side by side.
- Explain what a reader of your notebook should check in order to catch this trap, and why no error message appears.

### Render and Submit

Restart the kernel, run all cells in order, resolve errors, and save. Add a short
Markdown completion note, then save again. From `stat303-pandas-numpy-workflow` in
the terminal, run:

```text
quarto render activity07.ipynb --to html
```

Follow the [Quarto refresher](vscode_setup.ipynb#render-and-submit-with-quarto):
inspect the HTML and a copy opened outside the project folder. Check your name,
predictions, code, outputs, and explanations for A–E. Upload only `activity07.html`
to the Pandas and NumPy Workflow Canvas quiz; keep your notebook locally.

**HTML grading (16 points):** dtype repairs, coercion audit, and a justified
missing-value decision (3); row function and vectorized version, both timed, with
`np.allclose` verification and a correct ratio (4); non-`object` conversion, both
axis totals with the axis explained, broadcasting with the row-sum check, and a
correct matrix product with its order check (4); labeled DataFrame rebuilt with the
original index and columns, column assigned back, and a correct safety explanation
(2); a working demonstration of one trap, its fix, and the check a reader should
perform (2); name, readable report, and completion note (1).

## Extended Practice {#extended-practice}

Use a separate notebook beside `data/`. These longer exercises are additional
practice, not requirements for the Canvas quiz. Include your code, outputs, and
explanations.

Start that notebook with the cell below. It repeats the Stage 2 cleaning and the
Stage 4 calculation and builds the large table the last exercise needs, so your
notebook runs correctly from a restart without borrowing anything from the chapter
notebook.

```python
import numpy as np
import pandas as pd
from pathlib import Path

raw = pd.read_csv(Path('data') / 'store_transactions.csv')

analysis = raw.copy()
price_text = (analysis['unit_price']
              .str.replace('$', '', regex=False)
              .str.replace(',', '', regex=False))
analysis['unit_price'] = pd.to_numeric(price_text, errors='coerce')
analysis['units'] = pd.to_numeric(analysis['units'], errors='coerce')

incomplete = analysis['units'].isna() | analysis['unit_price'].isna()
analysis = analysis[~incomplete].copy()
analysis['units'] = analysis['units'].astype('int64')

units = analysis['units'].to_numpy()
price = analysis['unit_price'].to_numpy()
rate = np.select([units >= 50, units >= 20], [0.10, 0.05], default=0.0)
revenue = units * price * (1 - rate)
analysis['revenue'] = pd.Series(revenue, index=analysis.index)
analysis['billed'] = pd.Series(revenue + np.where(revenue < 75, 4.95, 0.0),
                               index=analysis.index)

rng = np.random.default_rng(303)
n = 100_000
big = pd.DataFrame({
    'store': rng.choice(['North', 'South', 'West'], size=n),
    'product': rng.choice(['Notebook', 'Pen', 'Folder', 'Marker'], size=n),
    'units': rng.integers(1, 80, size=n),
    'unit_price': rng.choice([5.0, 2.0, 3.0, 4.0], size=n),
})
```

### Per-Store Summaries with Boolean Masks

Using the `analysis` table that cell builds, compute the mean revenue for each
store two ways: once with a pandas Boolean mask per store, and once by converting
revenue and store labels to arrays and masking those. Compare the results with
`np.allclose`, then explain which version you would put in a report and why. Which
parts of the bookkeeping did you have to do by hand in the array version?

### Per-Record Thresholds

Flag every transaction whose revenue is in the top 20% *for its own store*. Build
one array of per-record thresholds by computing the 80th percentile within each
store's mask, then do the comparison itself as a single NumPy comparison of two
arrays. Confirm that the row order of the threshold array still matches the row
order of the revenue array, and explain what would go wrong if it did not.

**Hint — the method you need.** `np.percentile(values, 80)` returns the value
below which 80% of the observations fall, so a record is in the top 20% of its
store when its revenue is at least that store's own 80th percentile. The function
takes an array and one or more percentages between 0 and 100; it returns `nan` if
any input value is missing, so use `np.nanpercentile()` when that is a
possibility; and it accepts an `axis` argument when you want one threshold per row
or per column rather than one for the whole array. NumPy Fundamentals introduces
it briefly under
[More Summaries: Median, Percentiles, and Spread](numpy_fundamentals.ipynb#more-summaries-median-percentiles-and-spread),
and the [`numpy.percentile` reference](https://numpy.org/doc/stable/reference/generated/numpy.percentile.html)
documents every argument.

### A Conversion That Does Not Pay

Find a calculation on the 100,000-row `big` table where converting to NumPy makes
the complete workflow *slower* rather than faster, and demonstrate it with
`%timeit`. Candidates worth trying: an operation over a string column; converting
the whole mixed DataFrame rather than selected columns; or converting, computing
one cheap thing, and converting straight back. Explain where the time actually
went.

## Cheat Sheet

| Task | Code |
|---|---|
| **Stages 1–2: pandas** | |
| Read a CSV | `pd.read_csv(path)` |
| See what you actually got | `df.dtypes`, `df.shape`, `df.head()` |
| Text to numbers, unreadable values to `NaN` | `pd.to_numeric(s, errors='coerce')` |
| Strip a character from a text column | `s.str.replace('$', '', regex=False)` |
| Parse dates | `pd.to_datetime(s)` |
| Count and drop missing values | `df.isna().sum()`, `df.dropna(subset=['col'])` |
| Audit failed conversions | `raw.loc[converted.isna()]` |
| Restore an integer dtype after cleaning | `s.astype('int64')` |
| **Stage 3: convert** | |
| Selected numeric columns to a 2D array | `df[['a', 'b']].to_numpy()` |
| Confirm you did not get `object` | `arr.dtype` |
| **Stage 4: compute** | |
| Two outcomes, one condition | `np.where(cond, a, b)` |
| Many tiers | `np.select(conds, values, default=...)` |
| Collapse an axis | `arr.sum(axis=0)`, `arr.mean(axis=1)` |
| One factor per row | `arr / arr.sum(axis=1, keepdims=True)` |
| Matrix product | `mat @ vec`, `np.matmul(A, B)` |
| `NaN`-safe statistics | `np.nanmean(arr)`, `np.nansum(arr)` |
| **Stages 5–6: return and validate** | |
| Array back to a labeled Series | `pd.Series(values, index=df.index, name='...')` |
| Array back to a labeled table | `pd.DataFrame(arr, index=df.index, columns=df.columns)` |
| One value per row as a new column | `df['new'] = arr` (positional — check the row order) |
| Compare float results | `np.allclose(a, b)` |
| Compare exact results | `(a == b).all()`, `np.array_equal(a, b)` |
| **Stage 7: measure** | |
| Time one line in Jupyter | `%timeit expression` |
| Time a whole cell in Jupyter | `%%timeit` as the cell's first line |
| Time outside Jupyter | `time.perf_counter()` before and after |

## Before You Move On {#before-you-move-on}

The mental model to carry forward is the seven-stage pipeline: **pandas at both
ends, NumPy in the middle.** You load and clean in pandas, because only pandas
understands messy, labeled, mixed data. You compute in NumPy once the data has
become a rectangle of same-typed numbers, because that is what arrays are for. You
come back to pandas to put names on the results.

For any analysis that crosses between the two libraries, you should be able to
answer five questions without rerunning anything:

1. Which columns were converted, and were they genuinely numeric by then?
2. What happened to the incomplete records, and who decided?
3. What was the row and column order at conversion time, and is it still that order now?
4. Which labels does the result belong to, and how do you know?
5. Does the result match a reference you trust — on values, not just on shape?

Everything else is a trap to watch for: text columns masquerading as numbers,
`object` dtype, `NaN` semantics, and positional alignment. And the one performance
fact worth carrying is the one you measured in Stage 7 rather than assumed —
vectorizing beats looping row by row by orders of magnitude, while converting to
arrays on top of that is a tune-up.

Continue to [Data Visualization](Data%20visualization.ipynb), where these labeled,
validated results become figures.

References: [pandas and NumPy interoperability](https://pandas.pydata.org/docs/user_guide/basics.html#dtypes),
[`to_numpy()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_numpy.html),
[`to_numeric()`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html),
[NumPy broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html),
[`np.select`](https://numpy.org/doc/stable/reference/generated/numpy.select.html), and
[IPython `%timeit`](https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-timeit).